## **Assignment One Part One: Web Scrapping and Data Parsing**
**Student Name:** Jessica Mawuenam Dellason

**Student Number:** 25215734

This project will be based on [Dublin Rental Property Database](http://mlg.ucd.ie/modules/python/sources/rental/index.html)
### **Workflow Overview**

This section outlines the overall approach used for data collection before presenting the implementation details.

The workflow followed these main steps:

1. Inspect the website structure to identify relevant HTML tags and data fields.
2. Analyse how the URL changes between pages to handle pagination.
3. Scrape the required data from the website.
4. Parse the extracted HTML content and store the results as a list of dictionaries.
5. Save the structured data into a CSV file for later analysis.



---
#### **1. Setup and Initialisation**
In this section, the required Python libraries are imported and the main variables used throughout the scraping process are initialised.

The main webpage URL is defined, along with empty lists that will later store page links and the scraped data.

In [13]:
# Necessary imports and global variables
import urllib.request
import urllib.error
import bs4
import csv

main_link = "http://mlg.ucd.ie/modules/python/sources/rental/index.html"
# Stores links to each quarterly webpage found on the main page
quarter_links = []
# Stores the scraped listing data
listings = []
# Global mappings used to standardise categorical yes/no values
listing_yes = ["Yes", "Y", "True", "1", "yes"]
listing_no = ["No", "N", "False", "Not Available", "0", "no"]

----
#### **2. Reading the Main Webpage**
After inspecting the HTML structure of the main webpage, it was observed that the links to the quarterly listings are stored inside `<a>` tags using the `href` attribute.

The main page itself does not contain the rental data, but instead acts as an index linking to multiple quarter-specific pages.

This section retrieves the main webpage, parses the HTML content using BeautifulSoup, and extracts all links required for the next stage of scraping.

In [14]:
try:
    # Opening the link to the main webpage
    response = urllib.request.urlopen(main_link)

    # Reading the data from the main webpage 
    html = response.read().decode("utf-8")
    soup = bs4.BeautifulSoup(html, "html.parser")

    # Ensure quarter_links is empty before storing new links 
    if quarter_links:
        quarter_links.clear()

    # Getting the links to each quarter page
    for match in soup.find_all("a"):
        quarter_links.append(f"http://mlg.ucd.ie/modules/python/sources/rental/{match.get('href')}")

# Handling error
except urllib.error.HTTPError as e:
    print(f"HTTP Error {e.code}: Failed to retrieve {main_link}")
except urllib.error.URLError as e:
    print(f"Network Error: Failed to retrieve {main_link} - {e.reason}")
except Exception as e:
    print(f"Unexpected error retrieving {main_link}: {e}")

---
#### **3. Understanding the Website Structure**
Before implementing the scraper, the structure of the website was inspected using browser developer tools.

Each rental listing is contained within an `<li>` element.  
The listing header, stored in a `<span>` element with class `record` (e.g., `<span class="record">July 2025 — House</span>`), contains the month, year, and property type.  
Additional listing details are organised within nested table rows inside the same listing element.

Within these tables, attribute names (e.g., Price, Location, Bedrooms) appear inside `<td>` elements with class values `v1` or `v2`, while the corresponding values are stored in `<td>` elements without a class attribute.  

Understanding this structure was important for separating keys from values during extraction.

---
#### **4. URL Structure and Pagination Findings**
The listing pages follow a consistent and predictable URL structure:

`http://mlg.ucd.ie/modules/python/sources/rental/Q{quarter_number}-page{page_number:02}.html`

This structure contains two variable components:

- `quarter_number` identifies the quarter (e.g. Q1, Q2, Q3, Q4).
- `page_number` represents the page within that quarter.

The page number uses zero-padding (`:02`), meaning single-digit pages appear as `01`, `02`, etc., rather than `1`, `2`.

Because only these two values change while the rest of the URL remains fixed, different listing pages can be accessed by modifying the quarter and page number values.

It was observed that requesting a non-existent quarter or page number (e.g., [`http://mlg.ucd.ie/modules/python/sources/rental/Q3-page1.html`](http://mlg.ucd.ie/modules/python/sources/rental/Q3-page1.html)) results in the server returning a **Not Found** response, which raises an `HTTPError` exception.

The corresponding error page contains the text *"Not Found"* within an `<h1>` element, confirming that the requested resource does not exist.

---
#### **5. Applying the Findings to Data Extraction**

Based on the observed URL structure, the scraper iterates through quarterly listings and sequential page numbers to access all available data pages.

Pagination is implemented using an open-ended loop (`while True`) combined with exception handling. For each quarter, a page request is made using the expected URL pattern. If the request succeeds, the HTML content is retrieved and processed, and the page number is incremented to access the next page.

Each request is executed within a `try` block. When a non-existent page or quarter is requested, the server returns an HTTP error, which is captured by the corresponding exception handler. This behaviour naturally defines the termination condition for pagination, allowing the scraper to stop requesting additional pages once no further content exists and increments the quarter number by 1.

This approach avoids hard-coded page limits and ensures that all available pages are collected knowing that every year has 4 quarters while maintaining robust control over navigation errors.

---
#### **6. Parsing and Cleaning Listing Data**

After navigating to each listing page, the HTML content was parsed to extract structured information from individual rental entries.

##### - Extracting Month and Property Type
The `record` field combines multiple pieces of information in a single string. To make the data easier to analyse, the text was split around the separator (`—`) to isolate the date information from the property type.

The date portion was further split to extract only the month, since all listings belong to the same year (2025) and retaining the year would not provide additional analytical value. The property type was extracted as a separate variable.
***
##### - Price Extraction and Standardisation
Price values appeared in different textual formats, with some entries including additional text such as *"per month"*. Since rental prices are consistently monthly values, only the numeric component was required.

Non-numeric characters were removed while preserving the decimal separator, and the cleaned value was converted to a `float` type to maintain consistent numeric precision.
***
##### - Location and Postcode Cleaning
Location fields contained formatting inconsistencies, including different hyphen characters (`—`, `–`, and `-`). All variations were normalised to a single hyphen format before processing.

Where available, the location string was split into two components: the main location and the postcode. Some records contained only a location value, so conditional logic was applied to safely handle missing postcodes.

Postcode formatting was also standardised, as some entries used abbreviated forms (e.g., `D4`) while others used full representations (e.g., `Dublin 4`). Abbreviated values were converted to the full format to ensure consistency.
***
##### - Bedrooms and Bathrooms Conversion
The bedrooms and bathrooms fields occasionally contained additional text alongside numeric values. Since only the counts were required for analysis, non-numeric characters were removed and the remaining values were converted to integers.
***
##### - Parking and Garden Standardisation
The parking and garden fields contained multiple textual variations representing equivalent logical values (e.g., *Yes*, *Y*, *True*, *1*).

To ensure consistency, predefined groups of equivalent values were created and mapped to standardised outputs (`Yes` or `No`). Values that did not match known categories were assigned an empty value to safely handle unexpected entries.
***
##### - Lease Length Parsing and Missing Values
Since lease length represents a duration measured in months, the field was treated as numeric. Some records contained empty or missing values, which could cause conversion errors.

To handle this safely, conversion was performed inside a `try`–`except` block. When conversion failed, an empty value was assigned so that incomplete records could still be retained in the dataset.
***
##### - Contact Field Standardisation
The contact field contained categorical values such as *Owner* and *Estate Agent*, but capitalisation was inconsistent across records. The text was normalised using title case to ensure uniform formatting.
***
##### - Constructing Structured Listing Records
After extraction, all parsed values were combined into a dictionary representing a single listing. Each dictionary followed a consistent schema containing all required attributes.

These dictionaries were appended to a master list of listings, producing a structured dataset ready for export and further analysis.

In [17]:
# Ensure listing container is empty before scraping 
if listings:
    listings.clear()

# Iterate through each quarter
for quarter_number in range(1, len(quarter_links)+1):
    page_number = 1
        
    # Continue requesting pages until a missing page triggers an HTTP error
    while(True):
        try:
            content = bs4.BeautifulSoup(urllib.request.urlopen(f"http://mlg.ucd.ie/modules/python/sources/rental/Q{quarter_number}-page{page_number:02}.html").read().decode("utf-8"), "html.parser")

            # Extract all listing blocks from the page
            full_data = content.find_all("li")

            # Process each listing individually 
            for data in full_data:
                # Extract month and property type from the record field
                listing_month = data.find("span", {"class" : "record"}).text.strip().split("—")[0].split(" ")[0] 
                listing_type = data.find("span", {"class" : "record"}).text.strip().split("—")[1].strip()
                
                # Extract table values (cells without class attributes)
                data_items = [data_key.text for data_key in data.find_all("td",class_=lambda c: c is None) ] 

                # Clean and convert price to Decimal
                listing_price =  float("".join(numerical for numerical in data_items[0] if numerical.isdigit() or numerical == ".")) 
                
                # Normalise location formatting and split location/postcode
                full_address = data_items[1].strip().replace("—", "-").replace("–","-")
                if "-" in full_address:
                    listing_location = full_address.split("-")[0].strip()
                    listing_postcode = full_address.split("-")[1].strip()

                    # Standardise abbreviated postcode format
                    if len(listing_postcode) < 6:
                        listing_postcode = listing_postcode.replace("D", "Dublin ")

                else:
                    listing_location = full_address
                    listing_postcode = None
                
                    # Extract numeric bedroom and bathroom counts
                listing_bedrooms = int("".join(room for room in data_items[2] if room.strip().isdigit()))
                listing_bathrooms = int("".join(bath for bath in data_items[3] if bath.strip().isdigit()))

                # Standardise parking
                if data_items[4].strip() in listing_yes:
                    listing_parking = "Yes"
                elif data_items[4].strip() in listing_no:
                    listing_parking = "No"
                else:
                    listing_parking = None
                
                # Standardise garden
                if data_items[5].strip() in listing_yes:
                    listing_garden = "Yes"
                elif data_items[5].strip() in listing_no:
                    listing_garden = "No"
                else:
                    listing_garden = None

                # Convert lease length to integer where available 
                try: 
                    listing_lease = int("".join(lease for lease in data_items[6] if lease.strip().isdigit()))
                except ValueError:
                    listing_lease = None

                # Standardise contact field formatting
                listing_contact = data_items[7].strip().title()

                # Create structured listing record
                listing = {"Month": listing_month, 
                            "Type": listing_type, 
                            "Price": listing_price, 
                            "Location": listing_location, 
                            "Postcode": listing_postcode,
                            "Bedrooms": listing_bedrooms, 
                            "Bathrooms" : listing_bathrooms, 
                            "Parking": listing_parking, 
                            "Garden": listing_garden, 
                            "Lease Length": listing_lease, 
                            "Contact": listing_contact}
                
                # Store listing in master collection
                listings.append(listing)
        
            # Move to next page within the quarter
            page_number += 1
        
        
        # Stop pagination when page does not exist
        except urllib.error.HTTPError as e:
            print(f"HTTP Error {e.code}: Failed to retrieve 'http://mlg.ucd.ie/modules/python/sources/rental/Q{quarter_number}-page{page_number:02}.html'")
            break

        # Handle network-related errors
        except urllib.error.URLError as e:
            print(f"Network Error: Failed to retrieve 'http://mlg.ucd.ie/modules/python/sources/rental/Q{quarter_number}-page{page_number:02}.html' - {e.reason}")

        # Catch unexpected errors without stopping execution
        except Exception as e:
            print(f"Unexpected error retrieving 'http://mlg.ucd.ie/modules/python/sources/rental/Q{quarter_number}-page{page_number:02}.html': {e}")

HTTP Error 404: Failed to retrieve 'http://mlg.ucd.ie/modules/python/sources/rental/Q1-page27.html'
HTTP Error 404: Failed to retrieve 'http://mlg.ucd.ie/modules/python/sources/rental/Q2-page26.html'
HTTP Error 404: Failed to retrieve 'http://mlg.ucd.ie/modules/python/sources/rental/Q3-page24.html'
HTTP Error 404: Failed to retrieve 'http://mlg.ucd.ie/modules/python/sources/rental/Q4-page24.html'


---
#### **7. Saving the Dataset**
After all listings were parsed and stored as dictionaries, the data was exported to a CSV file for use in the next stage of the assignment.

A `DictWriter` was used to write the data, ensuring that dictionary keys were used as column headers and that each listing was saved as a structured row in the output file.

In [16]:
# Open CSV file in write mode
with open("data.csv", "w", newline="", encoding="utf-8") as data_file:

    # Create writer using dictionary keys as column headers
    writer = csv.DictWriter(data_file, fieldnames=listings[0].keys())

    # Write header row to CSV
    writer.writeheader()

    # Write each listing record as a row
    for row in listings:
        writer.writerow(row)

IndexError: list index out of range

---
### **Preview of Saved Dataset**
The table below shows a preview of the first 10 records from the saved data file.  
This confirms that the scraping and parsing steps produced a structured dataset with consistent columns and cleaned values, ready for analysis in the next notebook.

| Month   | Type      | Price | Location           | Postcode   | Bedrooms | Bathrooms | Parking | Garden | Lease Length | Contact       |
|---------|-----------|------:|--------------------|------------|---------:|----------:|---------|--------|-------------:|---------------|
| January | Apartment |   840 | Dublin City South  | Dublin 12  |        1 |         2 | Yes     | No     |           12 | Estate Agent  |
| January | House     |  3110 | Dublin City Nth.   | Dublin 1   |        3 |         1 | Yes     | Yes    |            6 | Owner         |
| January | Apartment |  2170 | Dublin City South  | Dublin 2   |        2 |         1 | No      |        |           12 | Estate Agent  |
| January | House     |  2580 | Dublin City Sth.   | Dublin 6   |        3 |         2 | Yes     | Yes    |           12 | Estate Agent  |
| January | Apartment |  1810 | Dublin City North  | Dublin 1   |        1 |         1 | No      | No     |           12 | Estate Agent  |
| January | Apartment |  2830 | Dublin City North  | Dublin 1   |        2 |         2 | No      | No     |           12 | Estate Agent  |
| January | House     |  3520 | Dublin City North  | Dublin 1   |        4 |         2 | No      | Yes    |            3 | Owner         |
| January | Apartment |  1650 | Dublin City North  | Dublin 1   |        1 |         1 | No      | No     |           12 | Estate Agent  |
| January | House     |  3280 | Dublin City South  | Dublin 22  |        3 |         2 | Yes     | Yes    |           12 | Owner         |